In [ ]:
from models.ML.ParliamentaryVectorization import ParliamentaryVectorization
from models.ML.PULKMeans import PULKMeans
from models.ML.PULKMRecommender import PULKMRecommender
import os

/home/miguel/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def test_full_pipeline():
    # 1. Configuración
    DATASET_PATH = "dataset/parcanDeb-rec-split"
    
    if not os.path.exists(DATASET_PATH):
        print(f"Error: No se encuentra {DATASET_PATH}")
        return

    # 2. Vectorización (El paso que revisamos antes)
    print("\n--- PASO 1: Vectorización ---")
    pv = ParliamentaryVectorization(DATASET_PATH)
    pv.load_and_vectorize()
    
    # 3. Selección de un Diputado de prueba
    # Vamos a buscar a alguien con un número decente de intervenciones para que sea interesante
    # Buscamos en el mapeo interno
    target_id = None
    target_name = None
    
    for mid, doc_indices in pv.mp_to_matrix_indices.items():
        if len(doc_indices) > 20: # Alguien con > 20 intervenciones
            target_id = mid
            target_name = pv.id_to_name[mid]
            break
    
    if target_id is None:
        # Fallback si son pocos datos
        target_id = list(pv.mp_to_matrix_indices.keys())[0]
        target_name = pv.id_to_name[target_id]

    print(f"\n--- PASO 2: Extracción de Matrices P y U ---")
    print(f"Objetivo: Diputado '{target_name}' (ID: {target_id})")
    
    try:
        P, U = pv.get_data_for_mp(target_id)
        print(f"Dimensiones Positivos (P): {P.shape}")
        print(f"Dimensiones Unlabeled (U): {U.shape}")
    except Exception as e:
        print(f"Error extrayendo matrices: {e}")
        return

    # 4. Ejecución de PUL-KMeans
    print(f"\n--- PASO 3: Algoritmo PUL-KMeans ---")
    # Instanciamos tu clase
    pul_km = PULKMeans(max_iter=10, verbose=True)
    
    # Entrenamos
    reliable_neg_indices = pul_km.fit(P, U)
    
    # 5. Análisis de Resultados
    n_total_u = U.shape[0]
    n_reliable_neg = len(reliable_neg_indices)
    n_potential_pos = n_total_u - n_reliable_neg # Los que se movieron al cluster positivo
    
    print("\n--- RESULTADOS DEL FILTRADO ---")
    print(f"Total documentos Unlabeled: {n_total_u}")
    print(f"Negativos Fiables detectados: {n_reliable_neg} ({ (n_reliable_neg/n_total_u)*100:.2f}% )")
    print(f"Posibles Positivos ocultos (ruido en U): {n_potential_pos} ({ (n_potential_pos/n_total_u)*100:.2f}% )")
    
    # Validación lógica simple
    if n_reliable_neg == 0:
        print(" [!] ALERTA: No se detectaron negativos fiables. ¿El centroide positivo atrajo todo?")
    elif n_reliable_neg == n_total_u:
        print(" [!] Info: Todos los unlabeled se consideraron negativos (K-means no movió nada).")
    else:
        print(" [OK] El algoritmo separó U en dos grupos exitosamente.")

In [4]:
test_full_pipeline()


--- PASO 1: Vectorización ---
--- Cargando TRAIN dataset desde: dataset/parcanDeb-rec-split ---
-> Procesando intervenciones para construir el corpus...
-> Corpus preparado con 38674 intervenciones individuales.
-> Entrenando TfidfVectorizer...
   [OK] Matriz generada. Dimensiones: (38674, 40330)
   (Intervenciones: 38674, Vocabulario: 40330)

--- PASO 2: Extracción de Matrices P y U ---
Objetivo: Diputado 'Rojas De León' (ID: 400)
Dimensiones Positivos (P): (475, 40330)
Dimensiones Unlabeled (U): (38199, 40330)

--- PASO 3: Algoritmo PUL-KMeans ---
  [PUL-KM] Inicio: 475 Positivos vs 38199 Unlabeled
  [PUL-KM] Fin. Detectados 22967 Negativos Fiables (de 38199 Unlabeled)

--- RESULTADOS DEL FILTRADO ---
Total documentos Unlabeled: 38199
Negativos Fiables detectados: 22967 (60.12% )
Posibles Positivos ocultos (ruido en U): 15232 (39.88% )
 [OK] El algoritmo separó U en dos grupos exitosamente.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

DATASET_PATH = "dataset/parcanDeb-rec-split"
clf_global = PULKMRecommender(DATASET_PATH, max_iter_pulk=10)
clf_global.run_full_pipeline()


=== FASE 1: Vectorización ===
--- Cargando TRAIN dataset desde: dataset/parcanDeb-rec-split ---
-> Procesando intervenciones para construir el corpus...
-> Corpus preparado con 38674 intervenciones individuales.
-> Entrenando TfidfVectorizer...
   [OK] Matriz generada. Dimensiones: (38674, 40330)
   (Intervenciones: 38674, Vocabulario: 40330)

=== FASE 2: Entrenamiento ===

=== Entrenando 454 Clasificadores SVM (One-vs-Rest) ===


Entrenando: 100%|██████████| 454/454 [07:30<00:00,  1.01it/s]



=== FASE 3: Validación (DEV) ===

=== Optimizando Umbral en DEV ===
 -> Mejor Umbral encontrado: -0.2143 (Micro F1 en Dev: 0.3789)

=== FASE 4: Evaluación Final (TEST) ===
Aplicando umbral optimizado (-0.2143) en Test...

--- Evaluando 962 documentos de Test (Multi-Label) ---

=== REPORTE DE EVALUACIÓN MULTI-LABEL ===
>> CLASIFICACIÓN (Threshold-based)
   Accuracy (Subset):   0.0613
   Micro F1-Score:      0.3968  <-- CLAVE PARA PAPER
   Macro F1-Score:      0.1376

>> RANKING / IR
   MAP:                 0.5538  <-- NUEVA MÉTRICA PRINCIPAL
   nDCG:                0.7363
   Recall@1 :           0.2722
   Recall@5 :           0.5835
   Recall@10:           0.7032
   Recall@20:           0.8066


{'Accuracy (Subset)': 0.061330561330561334,
 'Precision (Micro)': np.float64(0.39126662810873336),
 'Recall (Micro)': np.float64(0.4024390243902439),
 'F1-Score (Micro)': np.float64(0.3967741935483871),
 'Precision (Macro)': np.float64(0.19237399097605543),
 'Recall (Macro)': np.float64(0.15012106294239247),
 'F1-Score (Macro)': np.float64(0.1375741629877382),
 'MAP': np.float64(0.5538367647722425),
 'nDCG': np.float64(0.736330863459348),
 'Recall@1': 0.2721962346962346,
 'Recall@5': 0.5834632709632706,
 'Recall@10': 0.7031552156552148,
 'Recall@20': 0.8066193941193934}